In [1]:
import numpy as np
import pandas as pd
pd.options.display.max_columns = 100

%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sb

For file handling

In [3]:
import os, re, json, requests, yaml

## Biographical data

### From ProPublica

In [32]:
import requests, json

endpoint_s = "https://api.propublica.org/congress/v2/{congress}/{chamber}/members.json"

In [33]:
with open('propublica_key.txt') as f:
    key = f.readlines()[0]

In [ ]:
# Empty container to store results from the loop
congress_d = {}

# Loop through each of the Congresses
for _congress in range(111,119):
    
    # Print where we are in the loop
    print("{0}th Congress".format(_congress))
    
    # Format the endpoint_s URL with the number of congress
    _s = endpoint_s.format(congress=_congress,chamber='house')
    
    # Make the request and get the data
    _r = requests.get(_s,headers={'X-API-Key':key})
    
    # Turn the response data into a DataFrame
    _df = pd.DataFrame(_r.json()['results'][0]['members'])
    
    # Add a column to the DataFrame
    _df['congress'] = _congress
    
    # Store the DataFrame in the container
    congress_d[_congress] = _df

In [7]:
congress_df = pd.concat(congress_d.values(),ignore_index=True)
congress_df.to_csv('propublica_members.csv',index=False)
congress_df.head()

,id,title,short_title,api_uri,first_name,middle_name,last_name,suffix,date_of_birth,gender,party,leadership_role,twitter_account,facebook_account,youtube_account,govtrack_id,cspan_id,votesmart_id,icpsr_id,crp_id,google_entity_id,fec_candidate_id,url,rss_url,contact_form,in_office,cook_pvi,dw_nominate,ideal_point,seniority,total_votes,missed_votes,total_present,last_updated,ocd_id,office,phone,fax,state,district,at_large,geoid,missed_votes_pct,votes_with_party_pct,votes_against_party_pct,next_election,congress
0,A000014,Representative,Rep.,https://api.propublica.org/congress/v1/members...,Neil,None,Abercrombie,None,1938-06-26,M,D,None,neilabercrombie,None,hawaiirep1,400001,None,None,15244,None,/m/0255rj,,,None,None,False,None,None,None,22,1065.0,136.0,1.0,2019-10-07 10:50:00 -0400,ocd-division/country:us/state:hi/cd:1,None,None,None,HI,1,False,1501,12.77,98.48,1.41,NaN,111
1,A000022,Representative,Rep.,https://api.propublica.org/congress/v1/members...,Gary,L.,Ackerman,None,1942-11-19,M,D,None,repgaryackerman,None,RepAckerman,400003,1002061,None,15080,None,/m/03rhdq,,,None,None,False,None,None,None,28,1655.0,132.0,0.0,2019-10-07 10:50:00 -0400,ocd-division/country:us/state:ny/cd:5,None,None,None,NY,5,False,3605,7.98,98.75,1.19,2010,111
2,A000055,Representative,Rep.,https://api.propublica.org/congress/v1/members...,Robert,B.,Aderholt,None,1965-07-22,M,R,None,Robert_Aderholt,RobertAderholt,RobertAderholt,400004,45516,441,29701,N00003028,/m/024p03,,https://aderholt.house.gov,https://aderholt.house.gov/rss.xml,None,False,None,None,None,14,1655.0,63.0,0.0,2021-03-02 02:35:23 -0500,ocd-division/country:us/state:al/cd:4,None,None,None,AL,4,False,0104,3.81,93.37,6.56,2010,111
3,A000364,Representative,Rep.,https://api.propublica.org/congress/v1/members...,John,None,Adler,None,1959-08-23,M,D,None,None,None,None,412264,1030767,None,None,None,/m/05jzss,,,None,None,False,None,None,None,2,1655.0,34.0,0.0,2019-10-07 10:50:00 -0400,ocd-division/country:us/state:nj/cd:3,None,None,None,NJ,3,False,3403,2.05,89.70,10.24,2010,111
4,A000358,Representative,Rep.,https://api.propublica.org/congress/v1/members...,Todd,None,Akin,None,1947-07-05,M,R,None,reptoddakin,None,RepToddAkin,400005,87412,None,20123,None,/m/025xzm,,,None,None,False,None,None,None,10,1655.0,79.0,1.0,2019-10-07 10:50:00 -0400,ocd-division/country:us/state:mo/cd:2,None,None,None,MO,2,False,2902,4.77,94.66,5.27,2010,111


### congress-legislators

In [4]:
r = requests.get('https://github.com/unitedstates/congress-legislators/raw/refs/heads/main/legislators-historical.yaml')
legislators_yaml = yaml.safe_load(r.text)

In [84]:
pd.json_normalize(
    legislators_yaml[300],
    record_path=['terms'],          # one row per term
    meta=['id', 'name', 'bio'],   # carry these fields to each row
    sep='_'
).to_clipboard()

In [6]:
legislator_l = []

for l in legislators_yaml:
    legislator_l.append(pd.json_normalize(
        l,
        record_path=['terms'],
        meta= [
            ['id', 'bioguide'],
            ['id', 'govtrack'],
            ['id', 'icpsr'],
            ['id', 'house_history'],
            ['id', 'wikipedia'],
            ['id', 'wikidata'],
            ['id', 'google_entity_id'],
            ['name', 'first'],
            ['name', 'last'],
            ['bio', 'birthday'],
            ['bio', 'gender']
        ],
        sep='_',
        errors='ignore'
    )
                       )

legislator_df = pd.concat(legislator_l)

legislator_df.head(20)

,type,start,end,state,class,party,id_bioguide,id_govtrack,id_icpsr,id_house_history,id_wikipedia,id_wikidata,id_google_entity_id,name_first,name_last,bio_birthday,bio_gender,district,how,party_affiliations,caucus,url,address,phone,fax,contact_form,office,state_rank,rss_url,end-type
0,sen,1789-03-04,1793-03-03,DE,2.0,Anti-Administration,B000226,401222,507,NaN,Richard Bassett (Delaware politician),Q518823,kg:/m/02pz46,Richard,Bassett,1745-04-02,M,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,rep,1789-03-04,1791-03-03,VA,NaN,NaN,B000546,401521,786,9479,Theodorick Bland (congressman),Q1749152,kg:/m/033mf4,Theodorick,Bland,1742-03-21,M,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,rep,1789-03-04,1791-03-03,SC,NaN,NaN,B001086,402032,1260,10177,Aedanus Burke,Q380504,kg:/m/03yccv,Aedanus,Burke,1743-06-16,M,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,rep,1789-03-04,1791-03-03,MD,NaN,NaN,C000187,402334,1538,10687,Daniel Carroll,Q674371,kg:/m/02q22c,Daniel,Carroll,1730-07-22,M,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,rep,1789-03-04,1791-03-03,PA,NaN,NaN,C000538,402671,1859,11120,George Clymer,Q708913,kg:/m/01mpsj,George,Clymer,1739-03-16,M,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,rep,1789-03-04,1791-03-03,MD,NaN,NaN,C000710,402834,2010,11343,Benjamin Contee,Q868334,kg:/m/03xb7p,Benjamin,Contee,NaN,M,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,sen,1789-03-04,1791-03-03,MA,1.0,Pro-Administration,D000013,403156,2307,NaN,Tristram Dalton,Q1365791,kg:/m/03ynvb,Tristram,Dalton,1738-05-28,M,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,sen,1789-03-04,1791-03-03,NJ,1.0,Pro-Administration,E000155,403846,2943,12784,Jonathan Elmer,Q929830,kg:/m/03zxl_,Jonathan,Elmer,1745-11-29,M,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,sen,1789-03-04,1793-03-03,GA,2.0,Anti-Administration,F000100,404057,3128,13100,William Few,Q664099,kg:/m/02pz57,William,Few,1748-06-08,M,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,rep,1789-03-04,1791-03-03,NY,NaN,NaN,F000224,404179,3237,13251,William Floyd,Q1381060,kg:/m/01mpvf,William,Floyd,1734-12-17,M,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Disbursement data

Use the ProPublica "[House Office Expenditure Data](https://projects.propublica.org/represent/expenditures)" databases.

In [7]:
files = sorted([f for f in os.listdir('./disbursement_data/') if '.csv' in f])

In [8]:
disbursement_dfs = {}

for f in files:
    _quarter = f[:6]
    _df = pd.read_csv(
        './disbursement_data/' + f,
        encoding='latin1',
        low_memory=False,
        thousands=','
    )    
    
    _df.columns = [col.strip() for col in _df.columns]
        
    # Rename column
    if 'SORT SUBTOTAL DESCRIPTION' in _df.columns:
        _df = _df.rename(columns={'SORT SUBTOTAL DESCRIPTION':'CATEGORY'})

    # Filter and clean data direct from Congress
    if 'SORT SEQUENCE' in _df.columns:
    # Filter to details
        _df = _df[_df['SORT SEQUENCE'] == 'DETAIL']
        
        # Rename columns
        _df = _df.rename(columns={
            'TRANSACTION DATE':'DATE',
            'PERFORM START DT':'START DATE',
            'PERFORM END DT':'END DATE',
            'DESCRIPTION':'PURPOSE',
            'ORGANIZATION':'OFFICE',
            'VENDOR NAME':'PAYEE',
            'DOCUMENT':'RECORDID'
        })
        
    disbursement_dfs[_quarter] = _df

In [9]:
# Concatenate CSV files
all_db_df = pd.concat(disbursement_dfs,names=['YEAR-QUARTER']).reset_index(level=0)

# Extract and over-write inconsistent QUARTER and YEAR values
all_db_df['QUARTER'] = all_db_df['YEAR-QUARTER'].str.slice(4)
all_db_df['YEAR'] = all_db_df['YEAR-QUARTER'].str.slice(0,4).astype(int)

# Different category values
all_db_df.replace(
    {'CATEGORY':
     {'RENT, COMMUNICATION, UTILITIES':'RENT COMMUNICATION UTILITIES',
      'RENT  COMMUNICATION  UTILITIES':'RENT COMMUNICATION UTILITIES'}
    },
    inplace=True
)

all_db_df.reset_index(drop=True,inplace=True)

# Inspect
print(all_db_df.shape)
all_db_df.head()

(6419098, 27)


,YEAR-QUARTER,BIOGUIDE_ID,OFFICE,QUARTER,CATEGORY,DATE,PAYEE,START DATE,END DATE,PURPOSE,AMOUNT,YEAR,TRANSCODE,TRANSCODELONG,RECORDID,RECIP (orig.),PROGRAM,SORT SEQUENCE,DATA SOURCE,id,FISCAL YEAR OR LEGISLATIVE YEAR,ORGANIZATION CODE,PROGRAM CODE,BUDGET OBJECT CLASS,VENDOR ID,BUDGET OBJECT CODE,Unnamed: 18
0,2009Q3,NaN,COMMUNICATIONS,Q3,OTHER SERVICES,NaN,07ÃÂ­01 P2 OPR0900726A S...,10/04/06,10/04/06,NON-TECHNOLOGY SERVICE CONTRCT,16799.25,2009,NaN,NaN,NaN,07ÃÂ­01 P2 OPR0900726A S...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2009Q3,NaN,COMMUNICATIONS,Q3,OTHER SERVICES,NaN,07ÃÂ­22 P2 OPR0900726B ...,10/04/06,10/04/06,NON-TECHNOLOGY SERVICE CONTRCT,3876.75,2009,NaN,NaN,NaN,07ÃÂ­22 P2 OPR0900726B ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009Q3,NaN,COMMUNICATIONS,Q3,OTHER SERVICES,NaN,08ÃÂ­06 P2 FSS0000575A T...,07/18/06,07/18/06,NON-TECHNOLOGY SERVICE CONTRCT,2132.00,2009,NaN,NaN,NaN,08ÃÂ­06 P2 FSS0000575A T...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2009Q3,NaN,COMMUNICATIONS,Q3,OTHER SERVICES,NaN,08ÃÂ­25 P2 MFP0003163 A...,05/29/09,05/29/09,NON-TECHNOLOGY SERVICE CONTRCT,888.00,2009,NaN,NaN,NaN,08ÃÂ­25 P2 MFP0003163 A...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2009Q3,NaN,COMMUNICATIONS,Q3,OTHER SERVICES,NaN,09ÃÂ­10 P2 OPR0900726C S...,10/04/06,10/04/06,NON-TECHNOLOGY SERVICE CONTRCT,590.18,2009,NaN,NaN,NaN,09ÃÂ­10 P2 OPR0900726C S...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Exploring column values to help with cleanup and alignment

In [10]:
value_count_by_col = all_db_df.groupby(['YEAR']).agg({col:'nunique' for col in all_db_df.columns})
value_count_by_col

,YEAR-QUARTER,BIOGUIDE_ID,OFFICE,QUARTER,CATEGORY,DATE,PAYEE,START DATE,END DATE,PURPOSE,AMOUNT,YEAR,TRANSCODE,TRANSCODELONG,RECORDID,RECIP (orig.),PROGRAM,SORT SEQUENCE,DATA SOURCE,id,FISCAL YEAR OR LEGISLATIVE YEAR,ORGANIZATION CODE,PROGRAM CODE,BUDGET OBJECT CLASS,VENDOR ID,BUDGET OBJECT CODE,Unnamed: 18
YEAR,,,,,,,,,,,,,,,,,,,,,,,,,,,
2009,2,482,584,2,10,66,194153,1314,1844,2581,59932,1,0,0,0,194154,0,0,0,0,0,0,0,0,0,0,0
2010,4,458,673,4,10,211,215961,2159,3103,3697,96820,1,34,30,153059,215963,0,0,0,0,0,0,0,0,0,0,0
2011,4,545,652,4,10,289,30670,1679,2359,4527,94899,1,3,3,278562,30670,0,0,0,0,0,0,0,0,0,0,0
2012,4,507,617,4,10,266,28137,1800,2494,3717,88780,1,3,3,248835,28137,0,0,0,0,0,0,0,0,0,0,0
2013,4,538,639,4,10,272,30106,1059,1462,4077,85175,1,3,3,209444,30106,0,0,0,0,0,0,0,0,0,0,0
2014,4,471,566,4,10,260,28319,1013,1410,3739,83900,1,3,3,190392,28319,0,0,0,0,0,0,0,0,0,0,0
2015,4,515,612,4,10,280,29574,1107,1503,3911,84676,1,3,3,188929,29574,0,0,0,0,0,0,0,0,0,0,0
2016,4,548,640,4,12,285,39102,1687,2182,3831,82360,1,3,3,176201,39102,97,0,0,0,0,0,0,0,0,0,0
2017,3,507,1630,3,12,239,27371,1683,2176,3929,72081,1,4,0,128390,18824,113,1,0,0,0,0,0,0,0,0,0


Columns that work across years:
* YEAR-QUARTER
* BIOGUIDE_ID
* QUARTER
* CATEGORY
* DATE
* START DATE
* END DATE
* PURPOSE
* AMOUNT
* YEAR

Columns that vary in values suspiciously:
* OFFICE - jumps up in 2017
* PAYEE - drops in 2011

Columns that are missing values in some years:
* TRANSCODE - 2009
* TRANSCODELONG - 2009; starting 2017
* RECORDID - 2009
* RECIP (orig.) - starting 2018
* PROGRAM - 2009-2015
* SORT SEQUENCE - 2009-2017; 2017 also suspcious

Columns present only in one year:
* TRANSACTION DATE - 2018
* DATA SOURCE - 2018
* DOCUMENT - 2018
* id - 2020

### Variation and error in date formats

In [11]:
all_db_df.groupby(['YEAR-QUARTER'])['DATE'].apply(lambda x:x.str.len().value_counts()).unstack(1)

/var/folders/lr/v195xr617d32k2zdwwh30sb40000gn/T/ipykernel_68943/472701367.py:1: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  all_db_df.groupby(['YEAR-QUARTER'])['DATE'].apply(lambda x:x.str.len().value_counts()).unstack(1)


,3.0,5.0,7.0,8.0,9.0,10.0
YEAR-QUARTER,,,,,,
2009Q4,NaN,103310.0,NaN,NaN,NaN,NaN
2010Q2,NaN,112952.0,NaN,NaN,NaN,NaN
2010Q3,NaN,NaN,NaN,NaN,NaN,114000.0
2010Q4,NaN,86736.0,NaN,NaN,NaN,NaN
2011Q1,NaN,94985.0,NaN,NaN,NaN,NaN
2011Q2,NaN,100229.0,NaN,NaN,NaN,NaN
2011Q3,NaN,NaN,NaN,NaN,NaN,93080.0
2011Q4,NaN,90386.0,NaN,NaN,NaN,NaN
2012Q1,NaN,96702.0,NaN,NaN,NaN,NaN


### Values

In [ ]:
all_db_df['CATEGORY'].value_counts()

In [ ]:
all_db_df['PURPOSE'].value_counts()

Exclude things that are not AP, GL, AR.

In [ ]:
all_db_df['TRANSCODE'].value_counts()

### Suspicious variation in values

#### Office jumps up in 2017

It looks like starting in 2017 they add year information to the office title.

**RECOMMEND**: Extract year into a new column, otherwise keep "OFFICE" to only have member name

**AUDIT**: Examine prior period expenditures should not be allowed after some time period.

In [ ]:
offices_2017 = set(all_db_df.loc[all_db_df['YEAR'] == '2017','OFFICE'].unique())
offices_2016 = set(all_db_df.loc[all_db_df['YEAR'] == '2016','OFFICE'].unique())

In [ ]:
list(offices_2016)[:10]

In [ ]:
list(offices_2017 - offices_2016)[:10]

#### Payee drops in 2011

2009 and 2010 "PAYEE" field has data quality and probably needs to be broken into new columns. Do these other values map onto columns in other years? According to PDF: "DATE, (something), VOUCHER NO, PAYEE"

**RECOMMEND**: Wait to clean up on 2010, otherwise break up and move to appropriate columns.

In [ ]:
payees_2011 = set(all_db_df.loc[all_db_df['YEAR'] == '2011','PAYEE'].unique())
payees_2010 = set(all_db_df.loc[all_db_df['YEAR'] == '2010','PAYEE'].unique())

list(payees_2011 - payees_2010)[:10]

#### Questions
* Why a "STATIONERY" office?
* Why expenses from before 2017 in 2017 expenses?

### Columns missing values

#### TRANSCODELONG

**RECOMMEND**: Drop "TRANSCODELONG" because it duplicates "TRANSCODE"

In [ ]:
all_db_df.loc[all_db_df['YEAR'] == '2016','TRANSCODELONG'].unique()

In [ ]:
all_db_df.loc[all_db_df['YEAR'] == '2017','TRANSCODELONG'].unique()

#### RECIP (orig.)

There's not RECIP column in original data. Appears to be a duplicate column for PAYEE. There are no RECIPs that are not also PAYEEs.

**RECOMMEND**: Rename "RECIP (orig.)" to "PAYEE"

In [ ]:
all_db_df.loc[all_db_df['YEAR'] == '2018','RECIP (orig.)'].unique()

In [ ]:
recip_2017 = set(all_db_df.loc[all_db_df['YEAR'] == '2017','RECIP (orig.)'].unique())
payee_2017 = set(all_db_df.loc[all_db_df['YEAR'] == '2017','PAYEE'].unique())

In [ ]:
recip_2017 - payee_2017

####  PROGRAM
Starts in 2016, missing before, more starting 2017.

Heather: In other other federal program, programs are sub-categories for specific expense classes but some of these don't make sense.

**RECOMMEND**: Keep where it's present, nothing to be done for earlier

In [ ]:
program_2017 = set(all_db_df.loc[all_db_df['YEAR'] == '2017','PROGRAM'].unique())
program_2016 = set(all_db_df.loc[all_db_df['YEAR'] == '2016','PROGRAM'].unique())
program_2015 = set(all_db_df.loc[all_db_df['YEAR'] == '2015','PROGRAM'].unique())

In [ ]:
program_2017 - program_2016

In [ ]:
program_2016 - program_2017

#### SORT SEQUENCE

2017 data looks like mis-labeled dates. 2018 appears to be macro-expense categories.

**RECOMMEND**: Check 2017 dates for coverage, just keep "DETAIL"

In [ ]:
sort_sequence_2017 = set(all_db_df.loc[all_db_df['YEAR'] == '2017','SORT SEQUENCE'].unique())
sort_sequence_2018 = set(all_db_df.loc[all_db_df['YEAR'] == '2018','SORT SEQUENCE'].unique())

In [ ]:
'DETAIL' in sort_sequence_2017

In [ ]:
list(sort_sequence_2017)[:5]

In [ ]:
sort_sequence_2018

Check for detail, subtotal, grand total outside of SORT SEQUENCE before 2017.

In [ ]:
_df = disbursement_dfs['2016Q1']
_df.loc[_df['AMOUNT'] == 2576.71]

### Columns present only one year

#### TRANSACTION DATE

2017 data looks like mis-labeled dates. 2018 appears to be macro-expense categories.

**RECOMMEND**: Check 2018 dates for coverage/overlap.

In [ ]:
transaction_date_2017 = set(all_db_df.loc[all_db_df['YEAR'] == '2017','TRANSACTION DATE'].unique())
transaction_date_2018 = set(all_db_df.loc[all_db_df['YEAR'] == '2018','TRANSACTION DATE'].unique())

In [ ]:
list(transaction_date_2018)[:5]

#### DATA SOURCE

Only in 2018, appears to transaction codes.

In [ ]:
data_source_2018 = set(all_db_df.loc[all_db_df['YEAR'] == '2018','DATA SOURCE'].unique())
data_source_2018

In [ ]:
all_db_df.loc[all_db_df['YEAR'] == '2018','DATA SOURCE'].value_counts()

#### DOCUMENT

**RECOMMEND**: Rename 2018 DOCUMENT to voucher IDs.

In [ ]:
document_2018 = set(all_db_df.loc[all_db_df['YEAR'] == '2018','DOCUMENT'].unique())
list(document_2018)[:5]

#### id

Unclear: sequential integers only present in 2020.

**RECOMMEND**: Drop

In [ ]:
id_2018 = set(all_db_df.loc[all_db_df['YEAR'] == '2020','id'].unique())
list(id_2018)[:5]

## Cleanup

Schema: 
* YEAR-QUARTER
* BIOGUIDE_ID
* QUARTER
* CATEGORY
* DATE
* START DATE
* END DATE
* PURPOSE
* AMOUNT
* YEAR
* OFFICE
* PAYEE
* TRANSCODE
* VOUCHER

Actions:
* Rename "DOCUMENT" column from 2018 to "voucher ID"
* Drop "id" column from 2020
* Drop "SUBTOTAL" and "GRAND TOTAL" from "SORT SEQUENCE" after 2017

### 2009–2010 
Ignore until raw inputs cleaned.

In [7]:
cleaned_all_df = all_db_df.copy()[all_db_df['YEAR'] > 2010]

### 2011–2016

Add years to DATE column.

In [8]:
# Replace 2019 years that snuck into 2011Q3 and 2012Q3 data
yq2011q3 = cleaned_all_df['YEAR-QUARTER'] == '2011Q3'
cleaned_all_df.loc[yq2011q3,'DATE'] = cleaned_all_df.loc[yq2011q3,'DATE'].str.replace('2019','2011')

yq2012q3 = cleaned_all_df['YEAR-QUARTER'] == '2012Q3'
cleaned_all_df.loc[yq2012q3,'DATE'] = cleaned_all_df.loc[yq2012q3,'DATE'].str.replace('2019','2012')

In [9]:
_notnull = cleaned_all_df.loc[:,'DATE'].notnull()
_str5 = cleaned_all_df.loc[:,'DATE'].str.len() == 5

for _year in range(2011,2017):
    for _quarter in range(1,5):
        _yq = "{0}Q{1}".format(_year,_quarter)
        _c = cleaned_all_df['YEAR-QUARTER'] == _yq 
#         print(_yq)
        
        if _yq not in ['2011Q3','2012Q3','2016Q4']:

            cleaned_all_df.loc[_c & _notnull & _str5,'DATE'] = '{0}-'.format(_year) + cleaned_all_df.loc[_c & _notnull & _str5,'DATE']
            

### 2017

In [10]:
# 2017Q2 had three columns shifted
cleaned_all_df.loc[cleaned_all_df['YEAR-QUARTER'] == "2017Q2",'RECORDID'] = cleaned_all_df.loc[cleaned_all_df['YEAR-QUARTER'] == "2017Q2",'TRANSCODE']
cleaned_all_df.loc[cleaned_all_df['YEAR-QUARTER'] == "2017Q2",'TRANSCODE'] = cleaned_all_df.loc[cleaned_all_df['YEAR-QUARTER'] == "2017Q2",'DATE']
cleaned_all_df.loc[cleaned_all_df['YEAR-QUARTER'] == "2017Q2",'DATE'] = cleaned_all_df.loc[cleaned_all_df['YEAR-QUARTER'] == "2017Q2",'SORT SEQUENCE']
cleaned_all_df.loc[cleaned_all_df['YEAR-QUARTER'] == "2017Q2",'SORT SEQUENCE'] = np.nan


### 2018

Clean "TRANSACTION DATE"

In [11]:
cleaned_all_df.loc[all_db_df['YEAR-QUARTER'] == '2018Q2','DATE'] = cleaned_all_df.loc[all_db_df['YEAR-QUARTER'] == '2018Q2','TRANSACTION DATE']
cleaned_all_df.drop(columns='TRANSACTION DATE',inplace=True)


In [12]:
cleaned_all_df.loc[all_db_df['YEAR-QUARTER'] == '2018Q2','TRANSCODE'] = cleaned_all_df.loc[all_db_df['YEAR-QUARTER'] == '2018Q2','DATA SOURCE']
cleaned_all_df.drop(columns='DATA SOURCE',inplace=True)


In [13]:
cleaned_all_df.loc[all_db_df['YEAR-QUARTER'] == '2018Q2','RECORDID'] = cleaned_all_df.loc[all_db_df['YEAR-QUARTER'] == '2018Q2','DOCUMENT']
cleaned_all_df.drop(columns='DOCUMENT',inplace=True)


### 2019

Nothing.

### 2020
Drop "id"

In [14]:
cleaned_all_df.drop(columns='id',inplace=True)

### 2021
Nothing.

### 2022

2022Q4 added some new columns.

In [15]:
_cols = ['PROGRAM CODE','BUDGET OBJECT CLASS','VENDOR ID','BUDGET OBJECT CODE','FISCAL YEAR OR LEGISLATIVE YEAR','ORGANIZATION CODE']
cleaned_all_df.drop(columns=_cols,inplace=True)

In [16]:
cleaned_all_df.loc[cleaned_all_df['YEAR-QUARTER'] == '2022Q4','START DATE'] = cleaned_all_df.loc[cleaned_all_df['YEAR-QUARTER'] == '2022Q4','PERFORM START DT']
cleaned_all_df.drop(columns='PERFORM START DT',inplace=True)

cleaned_all_df.loc[all_db_df['YEAR-QUARTER'] == '2022Q4','END DATE'] = cleaned_all_df.loc[cleaned_all_df['YEAR-QUARTER'] == '2022Q4','PERFORM END DT']
cleaned_all_df.drop(columns='PERFORM END DT',inplace=True)


### 2023 & 2024

    YEAR-QUARTER
    BIOGUIDE_ID
    QUARTER
    CATEGORY
    DATE
    START DATE
    END DATE
    PURPOSE
    AMOUNT
    YEAR
    OFFICE
    PAYEE
    TRANSCODE
    VOUCHER


In [18]:
new_quarters = ['2023Q1','2023Q2','2023Q3','2023Q4','2024Q1','2024Q2','2024Q3','2024Q4']

# for q in new_quarters:
    

,ORGANIZATION,FISCAL YEAR OR LEGISLATIVE YEAR,ORGANIZATION CODE,PROGRAM,PROGRAM CODE,CATEGORY,BUDGET OBJECT CLASS,SORT SEQUENCE,TRANSACTION DATE,DATA SOURCE,DOCUMENT,VENDOR NAME,VENDOR ID,PERFORM START DT,PERFORM END DT,DESCRIPTION,BUDGET OBJECT CODE,AMOUNT
0,2023 OFFICE OF THE SPEAKER,LY2023,21SH202,GENERAL EXPENDITURES,EXPEN,PERSONNEL COMPENSATION,11,DETAIL,NaN,GL,,BAYLES CHRISTOPHER A.,,3-Jan-23,31-Mar-23,SHARED EMPLOYEE,1101,23266.67
1,2023 OFFICE OF THE SPEAKER,LY2023,21SH202,GENERAL EXPENDITURES,EXPEN,PERSONNEL COMPENSATION,11,DETAIL,NaN,GL,,BEDNAR MARK M,,3-Jan-23,30-Jan-23,DIGITAL COMMUNICATIONS DIRECTO,1101,14000.00
2,2023 OFFICE OF THE SPEAKER,LY2023,21SH202,GENERAL EXPENDITURES,EXPEN,PERSONNEL COMPENSATION,11,DETAIL,NaN,GL,,BEDNAR MARK M,,1-Feb-23,31-Mar-23,DIR OF STRATEGIC COMMUNICATION,1101,30000.00
3,2023 OFFICE OF THE SPEAKER,LY2023,21SH202,GENERAL EXPENDITURES,EXPEN,PERSONNEL COMPENSATION,11,DETAIL,NaN,GL,,BERTOLINI STEVEN R.,,3-Jan-23,28-Feb-23,STAFF ASSISTANT,1101,9666.67
4,2023 OFFICE OF THE SPEAKER,LY2023,21SH202,GENERAL EXPENDITURES,EXPEN,PERSONNEL COMPENSATION,11,DETAIL,NaN,GL,,BERTOLINI STEVEN R.,,1-Mar-23,31-Mar-23,OPERATIONS COORDINATOR,1101,5500.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138167,FISCAL YEAR 2022 CDN ENHANCE,FY2022,,CDN ENHANCE,93800,OTHER SERVICES,25,DETAIL,31-Mar-23,AP,1648109,GENERAL DYNAMICS INFORMATION TECH INC,3406,30-Jan-23,24-Feb-23,NON-TECHNOLOGY SERVICE CONTR,2502,7566.63
138171,FISCAL YEAR 2021 CDN ENHANCE,FY2021,,CDN ENHANCE,93800,RENT COMMUNICATION UTILITIES,23,DETAIL,7-Mar-23,AP,1640579,EQUINIX INC,30223,1-Nov-22,30-Nov-22,UTILITIES,2360,6297.86
138172,FISCAL YEAR 2021 CDN ENHANCE,FY2021,,CDN ENHANCE,93800,RENT COMMUNICATION UTILITIES,23,DETAIL,7-Mar-23,AP,1640580,EQUINIX INC,30223,1-Oct-22,31-Oct-22,UTILITIES,2360,5465.24
138173,FISCAL YEAR 2021 CDN ENHANCE,FY2021,,CDN ENHANCE,93800,RENT COMMUNICATION UTILITIES,23,DETAIL,7-Mar-23,AP,1640584,EQUINIX INC,30223,1-Dec-22,31-Dec-22,UTILITIES,2360,2807.31


### Multiple years

In [17]:
# Rename RECIP (orig.) to VOUCHER_ID
cleaned_all_df.rename(columns={'RECIP (orig.)':'VOUCHER_ID'},inplace=True)

In [18]:
# Drop redunant TRANSCODELONG column
cleaned_all_df.drop(columns=['TRANSCODELONG'],inplace=True)

In [19]:
# Drop rows that have totals in SORT SEQUENCE
no_totals = cleaned_all_df['SORT SEQUENCE'].isin(['SUBTOTAL','GRAND TOTAL FOR ORGANIZATION'])
cleaned_all_df.drop(cleaned_all_df.loc[no_totals].index,inplace=True)

# Drop the SORT SEQUENCE column
cleaned_all_df.drop(columns='SORT SEQUENCE',inplace=True)

In [20]:
# Drop "TOTALS" from "PURPOSE" columns
cleaned_all_df = cleaned_all_df[~cleaned_all_df['PURPOSE'].str.contains('TOTALS').fillna(False)]

In [21]:
# Replace three white spaces with NaN
cleaned_all_df.replace({'DATE':{'   ':np.nan}},inplace=True)

In [22]:
# Cast dates to ISO-8601
cleaned_all_df['DATE'] = pd.to_datetime(cleaned_all_df['DATE'])

In [28]:
yq_term_map = {}

for i,q in enumerate(sorted(cleaned_all_df['YEAR-QUARTER'].unique())):
    if i % 8 == 0:
        term = 1
        yq_term_map[q] = term
    else:
        term += 1
        yq_term_map[q] = term
        
cleaned_all_df['TERM_QUARTER'] = cleaned_all_df['YEAR-QUARTER'].map(yq_term_map)

In [29]:
yq_term_map

{'2011Q1': 1,
 '2011Q2': 2,
 '2011Q3': 3,
 '2011Q4': 4,
 '2012Q1': 5,
 '2012Q2': 6,
 '2012Q3': 7,
 '2012Q4': 8,
 '2013Q1': 1,
 '2013Q2': 2,
 '2013Q3': 3,
 '2013Q4': 4,
 '2014Q1': 5,
 '2014Q2': 6,
 '2014Q3': 7,
 '2014Q4': 8,
 '2015Q1': 1,
 '2015Q2': 2,
 '2015Q3': 3,
 '2015Q4': 4,
 '2016Q1': 5,
 '2016Q2': 6,
 '2016Q3': 7,
 '2016Q4': 8,
 '2017Q1': 1,
 '2017Q2': 2,
 '2017Q3': 3,
 '2017Q4': 4,
 '2018Q1': 5,
 '2018Q2': 6,
 '2018Q3': 7,
 '2018Q4': 8,
 '2019Q1': 1,
 '2019Q2': 2,
 '2019Q3': 3,
 '2019Q4': 4,
 '2020Q1': 5,
 '2020Q2': 6,
 '2020Q3': 7,
 '2020Q4': 8,
 '2021Q1': 1,
 '2021Q2': 2,
 '2021Q3': 3,
 '2021Q4': 4,
 '2022Q1': 5,
 '2022Q2': 6,
 '2022Q3': 7,
 '2022Q4': 8}

### Check

In [22]:
value_count_by_col = cleaned_all_df.groupby(['YEAR']).agg({col:'nunique' for col in cleaned_all_df.columns})
value_count_by_col

,YEAR-QUARTER,BIOGUIDE_ID,OFFICE,QUARTER,CATEGORY,DATE,PAYEE,START DATE,END DATE,PURPOSE,AMOUNT,YEAR,TRANSCODE,RECORDID,VOUCHER_ID,PROGRAM
YEAR,,,,,,,,,,,,,,,,
2011,4,545,652,4,10,288,30670,1679,2359,4527,94899,1,3,278562,30670,0
2012,4,507,617,4,10,266,28137,1800,2494,3717,88780,1,3,248835,28137,0
2013,4,538,639,4,10,272,30106,1059,1462,4077,85175,1,3,209444,30106,0
2014,4,471,566,4,10,260,28319,1013,1410,3739,83900,1,3,190392,28319,0
2015,4,515,612,4,10,280,29574,1107,1503,3911,84676,1,3,188929,29574,0
2016,4,548,640,4,12,285,39102,1687,2182,3831,82360,1,3,176201,39102,97
2017,4,509,1864,4,12,315,29516,1818,2450,4070,84387,1,4,172987,18824,116
2018,4,472,1393,4,12,306,29511,1099,1526,3787,82300,1,4,157733,0,127
2019,4,545,1465,4,12,299,42261,1139,1539,4720,86099,1,4,172770,0,129


## Write to disk

In [24]:
cleaned_all_df.to_csv('all_disbursements.csv',index=False,encoding='utf8')